# Pytorch Fundamentals

In [1]:
import sys

import torch
import numpy as np

---
## 1. Version & Device

In [2]:
print("PyTorch version:", torch.__version__)

# Is a GPU (CUDA = NVIDIA) available?
print("CUDA available :", torch.cuda.is_available())

# Apple silicon (M1/M2/M3...) uses "mps" instead of CUDA
print("MPS available  :", torch.backends.mps.is_available())

# Standard trick: pick the best device once, reuse it everywhere.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("We will use device :", device)

# Reproducibility: fix the random seed so random numbers repeat.
torch.manual_seed(42)
print("random with seed 42:", torch.rand(2))
torch.manual_seed(42)
print("same seed again    :", torch.rand(2))  # identical output

PyTorch version: 2.11.0+cpu
CUDA available : False
MPS available  : False
We will use device : cpu
random with seed 42: tensor([0.8823, 0.9150])
same seed again    : tensor([0.8823, 0.9150])


---
## 2. Creating Tensors
A tensor is PyTorch's n-dimensional array (like a NumPy array, but it can live on a GPU and remember gradients).

0D = scalar, 1D = vector, 2D = matrix, 3D+ = "tensor"

In [3]:
scalar = torch.tensor(7)                                    # 0D
vector = torch.tensor([1, 2, 3])                            # 1D
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])               # 2D (2 rows, 3 cols) (2, 3)
cube = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])   # 3D (2, 2, 2)
print("scalar:", scalar, "| ndim =", scalar.ndim)
print("vector:", vector, "| ndim =", vector.ndim)
print("matrix:\n", matrix, "\n ndim =", matrix.ndim)
print("cube shape:", cube.shape, "| ndim =", cube.ndim)

# A one-element tensor -> plain Python number with .item()
print("scalar.item():", scalar.item(), type(scalar.item()))

scalar: tensor(7) | ndim = 0
vector: tensor([1, 2, 3]) | ndim = 1
matrix:
 tensor([[1, 2, 3],
        [4, 5, 6]]) 
 ndim = 2
cube shape: torch.Size([2, 2, 2]) | ndim = 3
scalar.item(): 7 <class 'int'>


In [4]:
# --- filled tensors ---
print("zeros:\n", torch.zeros(2, 3))                  # all 0
print("\nones:\n", torch.ones(2, 3))                  # all 1
print("\nfull (all 9):\n", torch.full((2, 3), 9.0))
print("\nidentity:\n", torch.eye(3))                  # 1s on the diagonal
print("\nempty (garbage values, fast):\n", torch.empty(2, 2))

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])

ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

full (all 9):
 tensor([[9., 9., 9.],
        [9., 9., 9.]])

identity:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

empty (garbage values, fast):
 tensor([[1.5979e-11, 0.0000e+00],
        [0.0000e+00, 0.0000e+00]])


In [5]:
# --- ranges ---
print("arange(0,10,2):", torch.arange(0, 10, 2))    # like Python range
print("linspace(0,1,5):", torch.linspace(0, 1, 5))  # 5 evenly spaced points

arange(0,10,2): tensor([0, 2, 4, 6, 8])
linspace(0,1,5): tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


In [6]:
# --- random tensors ---
torch.manual_seed(0)
print("rand  (uniform 0..1):\n", torch.rand(2, 3))
print("\nrandn (normal, mean 0, std 1):\n", torch.randn(2, 3))
print("\nrandint (ints in [0,10)):   \n", torch.randint(0, 10, (2, 3)))
print("\nrandperm(5) (shuffled 0..4):", torch.randperm(5))

rand  (uniform 0..1):
 tensor([[0.4963, 0.7682, 0.0885],
        [0.1320, 0.3074, 0.6341]])

randn (normal, mean 0, std 1):
 tensor([[ 1.2645, -0.6874,  0.1604],
        [-0.6065, -0.7831,  1.0622]])

randint (ints in [0,10)):   
 tensor([[4, 3, 6],
        [9, 1, 4]])

randperm(5) (shuffled 0..4): tensor([4, 2, 3, 0, 1])


In [7]:
# --- "like" versions: copy the shape/dtype/device of another tensor ---
base = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("zeros_like:\n", torch.zeros_like(base))
print("\nones_like :\n", torch.ones_like(base))
print("\nrand_like :\n", torch.rand_like(base))

zeros_like:
 tensor([[0., 0.],
        [0., 0.]])

ones_like :
 tensor([[1., 1.],
        [1., 1.]])

rand_like :
 tensor([[0.1852, 0.3734],
        [0.3051, 0.9320]])


---
## 3. Tensor Attributes and dypes
The 3 attributes you will check ALL the time when debugging:

shape, dtype, device

In [8]:
x = torch.randn(3, 4)
print("shape :", x.shape)          # torch.Size([3, 4])
print("size():", x.size())         # same thing
print("dtype :", x.dtype)          # torch.float32 (default for floats)
print("device:", x.device)         # cpu
print("ndim  :", x.ndim)           # number of dimensions
print("numel :", x.numel())        # total number of elements (3*4=12)
print("shape[0], shape[-1]:", x.shape[0], x.shape[-1])

shape : torch.Size([3, 4])
size(): torch.Size([3, 4])
dtype : torch.float32
device: cpu
ndim  : 2
numel : 12
shape[0], shape[-1]: 3 4


In [9]:
# --- data types ---
# Python ints   -> torch.int64  (a.k.a. long)  <- used for token ids / labels
# Python floats -> torch.float32               <- used for weights / features
print("int tensor  :", torch.tensor([1, 2]).dtype)
print("float tensor:", torch.tensor([1.0, 2.0]).dtype)
print("bool tensor :", torch.tensor([True, False]).dtype)

# Choose dtype explicitly
print("float64:", torch.tensor([1, 2], dtype=torch.float64).dtype)
print("float16:", torch.tensor([1, 2], dtype=torch.float16).dtype)

int tensor  : torch.int64
float tensor: torch.float32
bool tensor : torch.bool
float64: torch.float64
float16: torch.float16


In [10]:
# Convert dtype with .to(), .float(), .long(), .int(), .double(), .bool()
a = torch.tensor([1, 2, 3])
print("int -> float:", a.float(), a.float().dtype)
print("float -> long:", torch.tensor([1.9, 2.1]).long())  # truncates!
print("using .to():", a.to(torch.float32).dtype)

# COMMON ERROR: mixing dtypes in some operations
# e.g. matmul between int64 and float32 raises an error -> cast first.
try:
    torch.tensor([[1, 2]]) @ torch.tensor([[1.0], [2.0]])
except RuntimeError as e:
    print("dtype mismatch error (expected):", str(e)[:70], "...")

int -> float: tensor([1., 2., 3.]) torch.float32
float -> long: tensor([1, 2])
using .to(): torch.float32
dtype mismatch error (expected): expected m1 and m2 to have the same dtype, but got: long int != float ...


---
## 4. Indexing and Slicing  (works just like NumPy / Python lists)

In [11]:
x = torch.arange(1, 13).reshape(3, 4)
print("x:\n", x)

print("\nx[0]       (first row)   :", x[0])
print("x[-1]      (last row)    :", x[-1])
print("x[1, 2]    (row 1, col 2):", x[1, 2])
print("x[:, 0]    (first column):", x[:, 0])
print("\nx[0:2]     (rows 0 and 1):\n", x[0:2])
print("\nx[:, 1:3]  (cols 1 and 2):\n", x[:, 1:3])
print("\nx[::2]     (every 2nd row):\n", x[::2])
print("\nx[..., -1] (last col, ellipsis = 'all other dims'):", x[..., -1])

x:
 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])

x[0]       (first row)   : tensor([1, 2, 3, 4])
x[-1]      (last row)    : tensor([ 9, 10, 11, 12])
x[1, 2]    (row 1, col 2): tensor(7)
x[:, 0]    (first column): tensor([1, 5, 9])

x[0:2]     (rows 0 and 1):
 tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])

x[:, 1:3]  (cols 1 and 2):
 tensor([[ 2,  3],
        [ 6,  7],
        [10, 11]])

x[::2]     (every 2nd row):
 tensor([[ 1,  2,  3,  4],
        [ 9, 10, 11, 12]])

x[..., -1] (last col, ellipsis = 'all other dims'): tensor([ 4,  8, 12])


In [12]:
# --- boolean masking: select by condition ---
mask = x > 6
print("mask (x > 6):\n", mask)
print("x[mask] (1-D result):", x[mask])

mask (x > 6):
 tensor([[False, False, False, False],
        [False, False,  True,  True],
        [ True,  True,  True,  True]])
x[mask] (1-D result): tensor([ 7,  8,  9, 10, 11, 12])


In [13]:
# --- fancy indexing: pass a list/tensor of indices ---
print("rows 0 and 2:\n", x[[0, 2]])

idx = torch.tensor([3, 0, 1])
print("\ncolumns in custom order:\n", x[:, idx])

rows 0 and 2:
 tensor([[ 1,  2,  3,  4],
        [ 9, 10, 11, 12]])

columns in custom order:
 tensor([[ 4,  1,  2],
        [ 8,  5,  6],
        [12,  9, 10]])


In [14]:
# --- modifying through indexing ---
y = x.clone()
y[0, 0] = 100
y[y > 10] = -1                 # set everything > 10 to -1
print("modified copy:\n", y)

modified copy:
 tensor([[-1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, -1, -1]])


In [15]:
# --- torch.where: if/else for tensors ---
print("where(x>6, x, 0):\n", torch.where(x > 6, x, torch.tensor(0)))

# --- gather helpers you will meet in NLP ---
print("\nindex_select rows [2,0]:\n", torch.index_select(x, 0, torch.tensor([2, 0])))

where(x>6, x, 0):
 tensor([[ 0,  0,  0,  0],
        [ 0,  0,  7,  8],
        [ 9, 10, 11, 12]])

index_select rows [2,0]:
 tensor([[ 9, 10, 11, 12],
        [ 1,  2,  3,  4]])


---
## 5. Reshaping and Joining tensors
Shape bugs are THE most common PyTorch bug. Learn these well.

In [16]:
x = torch.arange(12)                       # shape [12]
print("original:", x, x.shape)

# reshape / view: same data, new shape (total elements must match)
print("reshape(3,4):\n", x.reshape(3, 4))
print("reshape(2,-1)  (-1 = 'figure it out'):", x.reshape(2, -1).shape)
print("view(4,3):", x.view(4, 3).shape)    # view needs contiguous memory

original: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]) torch.Size([12])
reshape(3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
reshape(2,-1)  (-1 = 'figure it out'): torch.Size([2, 6])
view(4,3): torch.Size([4, 3])


In [17]:
# flatten: collapse to 1-D (or from a given dim)
m = torch.randn(2, 3, 4)
print("flatten all:", m.flatten().shape)
print("flatten from dim 1:", m.flatten(start_dim=1).shape)   # [2, 12]

flatten all: torch.Size([24])
flatten from dim 1: torch.Size([2, 12])


In [18]:
# unsqueeze adds a dimension of size 1, squeeze removes them
v = torch.tensor([1, 2, 3])                # [3]
print("unsqueeze(0):", v.unsqueeze(0).shape)   # [1, 3]  (row vector)
print("unsqueeze(1):", v.unsqueeze(1).shape)   # [3, 1]  (column vector)
print("v[None, :] is the same as unsqueeze(0):", v[None, :].shape)
print("squeeze:", torch.zeros(1, 3, 1).squeeze().shape)     # [3]

unsqueeze(0): torch.Size([1, 3])
unsqueeze(1): torch.Size([3, 1])
v[None, :] is the same as unsqueeze(0): torch.Size([1, 3])
squeeze: torch.Size([3])


In [19]:
# transpose / permute: reorder dimensions
a = torch.randn(2, 3)
print("a.T:", a.T.shape)                            # 2-D only
print("transpose(0,1):", a.transpose(0, 1).shape)
b = torch.randn(2, 3, 4)                            # e.g. [batch, seq, features]
print("permute(2,0,1):", b.permute(2, 0, 1).shape)  # [4, 2, 3]

# after transpose/permute memory is non-contiguous -> view() fails
t = a.T
print("is_contiguous after transpose:", t.is_contiguous())
try:
    t.view(6)
except RuntimeError:
    print("view() failed (expected). Fix: t.reshape(6) or t.contiguous().view(6)")
print("reshape works:", t.reshape(6))

a.T: torch.Size([3, 2])
transpose(0,1): torch.Size([3, 2])
permute(2,0,1): torch.Size([4, 2, 3])
is_contiguous after transpose: False
view() failed (expected). Fix: t.reshape(6) or t.contiguous().view(6)
reshape works: tensor([ 2.0576, -0.7663, -0.0354,  1.0993,  0.0627,  2.7565])


In [20]:
# --- joining ---
p = torch.tensor([[1, 2], [3, 4]])
q = torch.tensor([[5, 6], [7, 8]])
print("cat dim=0 (stack rows) :", torch.cat([p, q], dim=0).shape)     # [4, 2]
print("cat dim=1 (side by side):", torch.cat([p, q], dim=1).shape)    # [2, 4]
print("stack (NEW dim)         :", torch.stack([p, q], dim=0).shape)  # [2, 2, 2]
# cat = join along an EXISTING dim, stack = create a NEW dim

# --- splitting ---
z = torch.arange(10)
print("chunk into 5:", torch.chunk(z, 5))
print("split sizes [3,7]:", torch.split(z, [3, 7]))

# --- repeating ---
print("repeat:", torch.tensor([1, 2]).repeat(3))         # [1,2,1,2,1,2]
print("expand:", torch.tensor([[1], [2]]).expand(2, 3))  # broadcast copy

cat dim=0 (stack rows) : torch.Size([4, 2])
cat dim=1 (side by side): torch.Size([2, 4])
stack (NEW dim)         : torch.Size([2, 2, 2])
chunk into 5: (tensor([0, 1]), tensor([2, 3]), tensor([4, 5]), tensor([6, 7]), tensor([8, 9]))
split sizes [3,7]: (tensor([0, 1, 2]), tensor([3, 4, 5, 6, 7, 8, 9]))
repeat: tensor([1, 2, 1, 2, 1, 2])
expand: tensor([[1, 1, 1],
        [2, 2, 2]])


---
## 6. Math Operation and Reductions

In [21]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# --- element-wise (position by position) ---
print("a + b :", a + b)                # same as torch.add(a, b)
print("a - b :", a - b)
print("a * b :", a * b)                # element-wise, NOT matrix multiply
print("a / b :", a / b)
print("a ** 2:", a ** 2)
print("a + 10:", a + 10)               # scalar is applied to every element
print("sqrt, exp, log:", a.sqrt(), a.exp(), a.log())
print("abs, clamp(1.5, 2.5):", torch.tensor([-1.0, 2.0]).abs(), a.clamp(1.5, 2.5))

a + b : tensor([5., 7., 9.])
a - b : tensor([-3., -3., -3.])
a * b : tensor([ 4., 10., 18.])
a / b : tensor([0.2500, 0.4000, 0.5000])
a ** 2: tensor([1., 4., 9.])
a + 10: tensor([11., 12., 13.])
sqrt, exp, log: tensor([1.0000, 1.4142, 1.7321]) tensor([ 2.7183,  7.3891, 20.0855]) tensor([0.0000, 0.6931, 1.0986])
abs, clamp(1.5, 2.5): tensor([1., 2.]) tensor([1.5000, 2.0000, 2.5000])


In [22]:
# --- in-place ops end with underscore: they modify the tensor ---
c = a.clone()
c.add_(100)
print("in-place add_:", c)

# --- comparisons return bool tensors ---
print("a > 1.5:", a > 1.5, "| a == b:", a == b)
print("all/any:", (a > 0).all(), (a > 2).any())

in-place add_: tensor([101., 102., 103.])
a > 1.5: tensor([False,  True,  True]) | a == b: tensor([False, False, False])
all/any: tensor(True) tensor(True)


In [23]:
# --- matrix multiplication (VERY important) ---
A = torch.randn(2, 3)
B = torch.randn(3, 4)
print("A @ B shape:", (A @ B).shape)                 # [2,3]@[3,4] -> [2,4]
print("torch.matmul same :", torch.matmul(A, B).shape)
print("dot product:", torch.dot(a, b))               # 1-D only: 1*4+2*5+3*6 = 32
# Batched matmul: [batch,n,m] @ [batch,m,k] -> [batch,n,k]
print("batched:", (torch.randn(8, 2, 3) @ torch.randn(8, 3, 5)).shape)

A @ B shape: torch.Size([2, 4])
torch.matmul same : torch.Size([2, 4])
dot product: tensor(32.)
batched: torch.Size([8, 2, 5])


In [24]:
# --- reductions (collapse dimensions) ---
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("sum all       :", x.sum())
print("sum dim=0     :", x.sum(dim=0), " (collapses rows -> per column)")
print("sum dim=1     :", x.sum(dim=1), " (collapses cols -> per row)")
print("keepdim=True  :", x.sum(dim=1, keepdim=True).shape)   # [2,1] not [2]
print("mean/std/var  :", x.mean(), x.std(), x.var())
print("max/min       :", x.max(), x.min())

values, indices = x.max(dim=1)                # max returns (values, indices)

print("max per row   :", values, indices)
print("argmax/argmin :", x.argmax(), x.argmin())     # index in flattened tensor
print("argmax dim=1  :", x.argmax(dim=1))            # <- used for predictions!
print("cumsum dim=1  :", x.cumsum(dim=1))
print("topk(2)       :", torch.topk(torch.tensor([5., 1., 9., 3.]), k=2))
print("sort          :", torch.sort(torch.tensor([5., 1., 9., 3.]), descending=True))

sum all       : tensor(21.)
sum dim=0     : tensor([5., 7., 9.])  (collapses rows -> per column)
sum dim=1     : tensor([ 6., 15.])  (collapses cols -> per row)
keepdim=True  : torch.Size([2, 1])
mean/std/var  : tensor(3.5000) tensor(1.8708) tensor(3.5000)
max/min       : tensor(6.) tensor(1.)
max per row   : tensor([3., 6.]) tensor([2, 2])
argmax/argmin : tensor(5) tensor(0)
argmax dim=1  : tensor([2, 2])
cumsum dim=1  : tensor([[ 1.,  3.,  6.],
        [ 4.,  9., 15.]])
topk(2)       : torch.return_types.topk(
values=tensor([9., 5.]),
indices=tensor([2, 0]))
sort          : torch.return_types.sort(
values=tensor([9., 5., 3., 1.]),
indices=tensor([2, 0, 3, 1]))


---
## 7. Broadcasting
When shapes differ, PyTorch stretches size-1 dims automatically.

Rules (compare shapes from the RIGHT):
- two dims are compatible if they are equal, or one of them is 1

In [25]:
m = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])              # [2, 3]
row = torch.tensor([10, 20, 30])           # [3]   -> treated as [1, 3]
col = torch.tensor([[100], [200]])         # [2, 1]

print("matrix + row:\n", m + row)          # row added to every row
print("matrix + col:\n", m + col)          # col added to every column
print("col + row -> [2,3]:\n", col + row)  # both stretch (outer-sum style)

matrix + row:
 tensor([[11, 22, 33],
        [14, 25, 36]])
matrix + col:
 tensor([[101, 102, 103],
        [204, 205, 206]])
col + row -> [2,3]:
 tensor([[110, 120, 130],
        [210, 220, 230]])


In [26]:
# Normalizing each row by its own mean: classic broadcasting use
x = torch.randn(4, 5)
mean = x.mean(dim=1, keepdim=True)         # [4,1]  <- keepdim makes this work
print("centered rows mean ~0:", (x - mean).mean(dim=1))

# Incompatible shapes raise an error
try:
    torch.zeros(2, 3) + torch.zeros(4)
except RuntimeError as e:
    print("error (expected):", str(e)[:75])

centered rows mean ~0: tensor([ 0.0000e+00, -1.7881e-08,  4.7684e-08, -1.1921e-08])
error (expected): The size of tensor a (3) must match the size of tensor b (4) at non-singlet


---
## 8. Numpy INTEROP, CLONE, DETACH

In [27]:
arr = np.array([1.0, 2.0, 3.0])

# NumPy -> torch
t1 = torch.from_numpy(arr)
t2 = torch.tensor(arr)
arr[0] = 999
print("from_numpy sees change:", t1)
print("tensor() is a copy    :", t2)

from_numpy sees change: tensor([999.,   2.,   3.], dtype=torch.float64)
tensor() is a copy    : tensor([1., 2., 3.], dtype=torch.float64)


In [28]:
# torch -> NumPy
t = torch.tensor([1.0, 2.0, 3.0])
n = t.numpy()
t[0] = -5
print("numpy view changed:", n)

# If a tensor requires grad or is on GPU, first: .detach().cpu().numpy()
w = torch.tensor([1.0, 2.0], requires_grad=True)
print("safe conversion:", w.detach().cpu().numpy())

numpy view changed: [-5.  2.  3.]
safe conversion: [1. 2.]


In [29]:
# clone(): real copy of a tensor
a = torch.tensor([1, 2, 3])
b = a          # NOT a copy, same object
c = a.clone()  # real copy
a[0] = 100
print("b (alias):", b, "| c (clone):", c)

# tolist(): back to plain Python
print("tolist:", torch.tensor([[1, 2], [3, 4]]).tolist())

b (alias): tensor([100,   2,   3]) | c (clone): tensor([1, 2, 3])
tolist: [[1, 2], [3, 4]]


---
## 9. DEVICES  (CPU / GPU)

In [30]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

x = torch.randn(2, 3)
x = x.to(device)                 # move (returns a new tensor, so reassign!)
print("x is on:", x.device)

device: cpu
x is on: cpu


In [31]:
# Create directly on a device (faster than create-then-move)
y = torch.ones(2, 3, device=device)
print("x + y works because both are on", device, ":", (x + y).shape)

# Back to CPU (needed before .numpy() or plotting)
print("back to cpu:", x.cpu().device)

# Mismatch error (only demonstrable if you have a GPU)
if device != "cpu":
    try:
        torch.ones(2, device="cpu") + torch.ones(2, device=device)
    except RuntimeError:
        print("device mismatch error (expected)")

x + y works because both are on cpu : torch.Size([2, 3])
back to cpu: cpu


---
## 10. AUTOGRAD  -- automatic differentiation
- Training a model = "compute a loss, get the gradient of the loss with
- respect to each parameter, nudge the parameter". PyTorch does the
- gradient part for you. That is autograd.

In [32]:
# --- basic example: y = x^2  ->  dy/dx = 2x ---
x = torch.tensor(3.0, requires_grad=True)   # "track operations on me"
y = x ** 2
print("y:", y)                              # has grad_fn -> it is in a graph
print("y.grad_fn:", y.grad_fn)
y.backward()                                # computes dy/dx
print("dy/dx at x=3 :", x.grad)             # 2*3 = 6

# --- a slightly longer chain: z = (2x + 1)^3  ->  dz/dx = 6 (2x+1)^2 ---
x = torch.tensor(2.0, requires_grad=True)
z = (2 * x + 1) ** 3
z.backward()
print("dz/dx at x=2 :", x.grad, "(expected 6*25 = 150)")

y: tensor(9., grad_fn=<PowBackward0>)
y.grad_fn: <PowBackward0 object at 0x7b770cc659c0>
dy/dx at x=3 : tensor(6.)
dz/dx at x=2 : tensor(150.) (expected 6*25 = 150)


In [33]:
# --- several parameters: like weights and bias ---
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x_in, target = torch.tensor(3.0), torch.tensor(10.0)
pred = w * x_in + b                         # 7
loss = (pred - target) ** 2                 # 9
loss.backward()
print("dloss/dw:", w.grad, " dloss/db:", b.grad)
# dloss/dw = 2*(pred-target)*x = 2*(-3)*3 = -18 ; dloss/db = -6

dloss/dw: tensor(-18.)  dloss/db: tensor(-6.)


In [34]:
# --- gradients ACCUMULATE: you must reset them ---
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (w * 2).backward()
    print(f"  step {i}: w.grad = {w.grad}")   # 2, 4, 6 (accumulating!)
w.grad.zero_()                                # reset (optimizers do this for you)
print("after zero_:", w.grad)

  step 0: w.grad = 2.0
  step 1: w.grad = 4.0
  step 2: w.grad = 6.0
after zero_: tensor(0.)


In [35]:
# --- backward() on a non-scalar needs a scalar: reduce first ---
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = (v ** 2).sum()                          # scalar
out.backward()
print("grad of sum(v^2):", v.grad)            # 2v = [2, 4, 6]

grad of sum(v^2): tensor([2., 4., 6.])


In [36]:
# --- turning tracking OFF (evaluation / inference) ---
w = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():                         # nothing inside is tracked
    out = w * 3
print("inside no_grad -> requires_grad:", out.requires_grad)   # False

# detach(): cut a tensor away from the graph
d = (w * 3).detach()
print("detached requires_grad:", d.requires_grad)

inside no_grad -> requires_grad: False
detached requires_grad: False


In [37]:
# requires_grad_() switches tracking on in-place
t = torch.ones(2)
t.requires_grad_()
print("requires_grad_:", t.requires_grad)

# --- only float tensors can require gradients ---
try:
    torch.tensor([1, 2, 3], requires_grad=True)
except RuntimeError as e:
    print("int tensors can't have grads (expected):", str(e)[:60])

requires_grad_: True
int tensors can't have grads (expected): Only Tensors of floating point and complex dtype can require


In [38]:
# --- leaf tensors: those YOU create with requires_grad=True get .grad ---
a = torch.tensor(2.0, requires_grad=True)   # leaf
b = a * 3                                   # NOT a leaf (result of an op)
print("a.is_leaf:", a.is_leaf, "| b.is_leaf:", b.is_leaf)

a.is_leaf: True | b.is_leaf: False


---
## 11. NLP-FLAVOURED TENSOR EXAMPLES
- All done with plain tensors, no neural network layers yet.

In [39]:
# --- 1. Text becomes integer ids (token ids). Always int64 (long). ---
vocab = {"<pad>": 0, "i": 1, "love": 2, "nlp": 3, "pytorch": 4, "hate": 5}
sentences = [["i", "love", "nlp"],
                ["i", "love", "pytorch", "nlp"],
                ["i", "hate", "nlp"]]

# --- 2. Pad to the same length so we can stack into ONE tensor ---
max_len = max(len(s) for s in sentences)
ids = [[vocab[w] for w in s] + [vocab["<pad>"]] * (max_len - len(s)) for s in sentences]
batch = torch.tensor(ids)                    # shape [batch=3, seq_len=4] 
print("batch of token ids:\n", batch, "\nshape:", batch.shape, batch.dtype)

batch of token ids:
 tensor([[1, 2, 3, 0],
        [1, 2, 4, 3],
        [1, 5, 3, 0]]) 
shape: torch.Size([3, 4]) torch.int64


In [40]:
# --- 3. Attention/padding mask: 1 = real token, 0 = padding ---
mask = (batch != vocab["<pad>"])
print("mask:\n", mask.int())
print("real lengths:", mask.sum(dim=1))

mask:
 tensor([[1, 1, 1, 0],
        [1, 1, 1, 1],
        [1, 1, 1, 0]], dtype=torch.int32)
real lengths: tensor([3, 4, 3])


In [41]:
# --- 4. Embedding lookup is just indexing into a matrix! ---
# (nn.Embedding, which you'll meet later, does exactly this + learns the matrix)
torch.manual_seed(1)
embedding_matrix = torch.randn(len(vocab), 5) 
embedded = embedding_matrix[batch] 
print("embedded shape [batch, seq, emb_dim]:", embedded.shape)

# --- 5. Masked mean pooling: average word vectors ignoring padding ---
m = mask.unsqueeze(-1).float()
sentence_vecs = (embedded * m).sum(dim=1) / m.sum(dim=1)
print("sentence vectors shape:", sentence_vecs.shape)

embedded shape [batch, seq, emb_dim]: torch.Size([3, 4, 5])
sentence vectors shape: torch.Size([3, 5])


In [42]:
# --- 6. Softmax: scores -> probabilities that sum to 1 ---
logits = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 0.3]])
probs = torch.softmax(logits, dim=-1)
print("softmax:\n", probs, "\nrow sums:", probs.sum(dim=-1))
print("predicted classes (argmax):", probs.argmax(dim=-1))

# --- 7. Cosine similarity between sentence vectors ---
cos = torch.nn.functional.cosine_similarity(sentence_vecs[0], sentence_vecs[1], dim=0)
print("cosine(sent0, sent1):", cos.item())

softmax:
 tensor([[0.6590, 0.2424, 0.0986],
        [0.1086, 0.8025, 0.0889]]) 
row sums: tensor([1.0000, 1.0000])
predicted classes (argmax): tensor([0, 1])
cosine(sent0, sent1): 0.8941132426261902


In [43]:
# --- 8. Scaled dot-product attention (the core of Transformers) ---
# Attention(Q,K,V) = softmax(Q K^T / sqrt(d)) V   -- only matmul + softmax!
d = 8
Q = torch.randn(1, 4, d)
K = torch.randn(1, 4, d)
V = torch.randn(1, 4, d)
scores = Q @ K.transpose(-2, -1) / d ** 0.5
weights = torch.softmax(scores, dim=-1)
output = weights @ V 
print("attention weights shape:", weights.shape, "| output shape:", output.shape)

# Masking padded positions in attention: put -inf BEFORE softmax
pad_mask = torch.tensor([[1, 1, 1, 0]]).bool()
scores = scores.masked_fill(~pad_mask[:, None, :], float("-inf"))
print("attention weights on padded key (should be 0):",
        torch.softmax(scores, dim=-1)[0, 0, -1].item())

attention weights shape: torch.Size([1, 4, 4]) | output shape: torch.Size([1, 4, 8])
attention weights on padded key (should be 0): 0.0


---
## 12. LINEAR REGRESSION FROM SCRATCH
Everything you learned so far, combined. NO nn module, NO optimizer:
- just tensors + autograd + a manual gradient-descent update.
- This is exactly what nn / optim will automate later.

In [44]:
torch.manual_seed(42)

# Fake data:  y = 3x + 2 + noise
X = torch.rand(100, 1) * 10
y = 3 * X + 2 + torch.randn(100, 1) * 0.5

# Parameters we want to LEARN (start random / zero)
w = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.01           # learning rate = step size
epochs = 300

for epoch in range(epochs):
    # 1) forward pass: make predictions
    y_pred = X @ w + b

    # 2) loss: how wrong are we? (mean squared error)
    loss = ((y_pred - y) ** 2).mean()

    # 3) backward pass: compute gradients of loss w.r.t. w and b
    loss.backward()

    # 4) update parameters (no_grad: the update itself must not be tracked)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # 5) reset gradients (otherwise they accumulate)
    w.grad.zero_()
    b.grad.zero_()

    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | "
                f"w {w.item():.3f} | b {b.item():.3f}")

print(f"\nLearned: y = {w.item():.2f}x + {b.item():.2f}   (true: y = 3x + 2)")
# The recipe above (forward -> loss -> backward -> update -> zero grad)
# is THE training loop. It will be the same for a giant transformer.

epoch   0 | loss 736.7183 | w 1.934 | b 0.482
epoch  50 | loss 0.4859 | w 3.145 | b 0.976
epoch 100 | loss 0.3641 | w 3.109 | b 1.218
epoch 150 | loss 0.2886 | w 3.081 | b 1.409
epoch 200 | loss 0.2419 | w 3.059 | b 1.560
epoch 250 | loss 0.2129 | w 3.041 | b 1.678
epoch 299 | loss 0.1953 | w 3.028 | b 1.769

Learned: y = 3.03x + 1.77   (true: y = 3x + 2)


---
## 14. SAVING AND LOADING TENSORS

In [45]:
import os
import tempfile

x = torch.randn(3, 3)
path = os.path.join(tempfile.gettempdir(), "tensor_demo.pt")

torch.save(x, path)
loaded = torch.load(path)
print("equal after reload:", torch.equal(x, loaded))

# Save several things in a dictionary (this is how model checkpoints work)
torch.save({"weights": x, "epoch": 5, "note": "demo"}, path)
ckpt = torch.load(path)
print("checkpoint keys:", list(ckpt.keys()))
os.remove(path)

equal after reload: True
checkpoint keys: ['weights', 'epoch', 'note']
